<a href="https://colab.research.google.com/github/Bokenasubi-T/pomacea-distribution-jp/blob/main/notebooks/08_convert_landuse_to_grid.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#08_Convert Land-use data into raster data
Link the land-use data to 30-sec grids and save it as raster data.

*   Input : landuse_100m_2021.parquet
*   Output item : ***



In [2]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd

# Project folders on Google Drive
BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/pomacea"
DATA_DIR = f"{BASE_DIR}/data"

Mounted at /content/drive


In [24]:
# read land-use data(parquet file)
landuse_100m_2021 = pd.read_parquet(f"{DATA_DIR}/landuse_100m_2021.parquet")
# check data
landuse_100m_2021.head(5)

,L03b_001,L03b_002
0,3622572688,0600
1,3622572689,0600
2,3622572692,0600
3,3622572696,0600
4,3622572697,0600


In [25]:
# calculate coordinate of the centre of mesh
# tear apart mesh-code for calculation
meshcode12 = landuse_100m_2021["L03b_001"] // 10**8
meshcode34 = (landuse_100m_2021["L03b_001"] // 10**6) % 100
meshcode5 = (landuse_100m_2021["L03b_001"] // 10**5) % 10
meshcode6 = (landuse_100m_2021["L03b_001"] // 10**4) % 10
meshcode7 = (landuse_100m_2021["L03b_001"] // 10**3) % 10
meshcode8 = (landuse_100m_2021["L03b_001"] // 10**2) % 10
meshcode9 = (landuse_100m_2021["L03b_001"] // 10) % 10
meshcode10 = landuse_100m_2021["L03b_001"] % 10

# calculate coordinate of the centre of mesh
landuse_100m_2021["lat"] = meshcode12 / 1.5 + meshcode5 * 5/60 + meshcode7 * 30/3600 + meshcode9 * 3/3600 + 1.5/3600
landuse_100m_2021["lon"] = meshcode34 +100 + meshcode6 * 7.5/60 + meshcode8 * 45/3600 + meshcode10 * 4.5/3600 + 2.25/3600

# delete variables for calculating coordinate since it require memory space a lot
del meshcode12, meshcode34, meshcode5, meshcode6, meshcode7, meshcode8, meshcode9, meshcode10

# check data
landuse_100m_2021.head(5)

,L03b_001,L03b_002,lat,lon
0,3622572688,0600,24.440417,122.960625
1,3622572689,0600,24.440417,122.961875
2,3622572692,0600,24.441250,122.953125
3,3622572696,0600,24.441250,122.958125
4,3622572697,0600,24.441250,122.959375


In [26]:
# install rasterio
!pip install rasterio

In [27]:
# get raster information
import rasterio

src_path = f"{DATA_DIR}/wc2.1_30s_bio_6_japan_land.tif"
# get width, height, west-end coordinate, north-end coordinate of raster data
with rasterio.open(src_path) as src:
    raster_coordi = src.transform
    raster_width = src.width
    raster_height = src.height

# calculate raster row and col number for centre of the land-use mesh
landuse_100m_2021["row"] = ((landuse_100m_2021["lat"] - raster_coordi.f) // raster_coordi.e).astype("int32")
landuse_100m_2021["col"] = ((landuse_100m_2021["lon"] - raster_coordi.c) // raster_coordi.a).astype("int32")

# check result
display(landuse_100m_2021.head(5))
# check the coverage of mesh within raster range
print("\n")
print("- Check the range of raster coverage")
print(f"max of row :{landuse_100m_2021["row"].max()} , min of row :{landuse_100m_2021["row"].min()} , number of row :{raster_height - 1}")
print(f"max of col :{landuse_100m_2021["col"].max()} , min of col :{landuse_100m_2021["col"].min()} , number of col :{raster_width - 1}")


,L03b_001,L03b_002,lat,lon,row,col
0,3622572688,0600,24.440417,122.960625,2530,4
1,3622572689,0600,24.440417,122.961875,2530,4
2,3622572692,0600,24.441250,122.953125,2530,3
3,3622572696,0600,24.441250,122.958125,2530,3
4,3622572697,0600,24.441250,122.959375,2530,4




- Check the range of raster coverage
max of row :2577 , min of row :-4 , number of row :2577
max of col :3727 , min of col :1 , number of col :2768


In [28]:
# check the mesh outside raster range
outside = (landuse_100m_2021["row"] < 0 ) | (landuse_100m_2021["col"] >= raster_width) | (landuse_100m_2021["col"] < 0) | (landuse_100m_2021["row"] >= raster_height) # set the condition to pick mesh outside raster range
# count the number of mesh outside raster range
print(f"Number of mesh outside range: {outside.sum()}")
# check the meshcode of mesh outside range
display((landuse_100m_2021[outside]["L03b_001"] // 10**6).value_counts())
# get rid of mesh outside range
landuse_100m_2021 = landuse_100m_2021[~outside]
print(f"Number of mesh inside range: {len(landuse_100m_2021)}")




Number of mesh outside range: 461440


,count
L03b_001,
6747,174929
6646,92918
6848,66628
6748,57637
6546,35589
6647,23628
6847,9986
3653,125


Number of mesh inside range: 36667206
